# 01. 벡터스토어 구축 (Ingest Pipeline)

이 노트북은 문서 폴더에서 파일을 읽어 ChromaDB 벡터스토어를 구축하는 전체 파이프라인을 단계별로 실행한다.

## 처리 순서
1. 설정 파일 로드
2. 문서 수집 (document_loader)
3. 텍스트 파싱 (parser_service)
4. 청킹 (chunk_service)
5. 임베딩 생성 (embedding_service)
6. ChromaDB 저장 (vector_store_service)

## 실행 전 준비사항
- `config/config.yaml` 설정 확인
- `data/doc/` 폴더에 처리할 문서 파일 배치
- 필요 패키지 설치: `pip install sentence-transformers chromadb`

In [ ]:
# 프로젝트 루트를 Python 경로에 추가 (노트북 위치가 프로젝트 루트 하위이므로 필요)
import sys
from pathlib import Path

# 노트북 위치: notebooks/ingest/ → 프로젝트 루트는 2단계 위
project_root = Path.cwd().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"프로젝트 루트: {project_root}")

## 1단계: 설정 파일 로드

In [ ]:
from src.db_to_llm.shared.config.config_loader import load_config
from src.db_to_llm.shared.logging.logger import setup_logger

config_path = project_root / "config" / "config.yaml"
config = load_config(config_path)
setup_logger(log_level=config.get("logging", {}).get("level", "INFO"))

ingest_config = config.get("ingest", {})
retrieval_config = config.get("retrieval", {})

print("설정 로드 완료")
print(f"  doc_dir: {ingest_config.get('doc_dir')}")
print(f"  parser: {ingest_config.get('parser')}")
print(f"  chunk_size: {ingest_config.get('chunk_size')}")
print(f"  embedding_model: {retrieval_config.get('embedding_model')}")
print(f"  chroma_path: {retrieval_config.get('chroma_path')}")

## 2단계: 문서 수집

In [ ]:
from src.db_to_llm.ingest.document_loader import collect_documents

doc_dir = project_root / ingest_config.get("doc_dir", "data/doc")
supported_extensions = ingest_config.get("supported_extensions", [".pdf", ".txt", ".sql", ".md"])

documents = collect_documents(doc_dir=doc_dir, supported_extensions=supported_extensions)

print(f"수집된 문서: {len(documents)}개")
for doc in documents:
    print(f"  - {doc.file_name} ({doc.file_type}, {doc.file_size} bytes)")

## 3단계: 텍스트 파싱

In [ ]:
from src.db_to_llm.ingest.parser_service import parse_documents

parser_name = ingest_config.get("parser", "simple")
parsed_documents = parse_documents(documents=documents, parser_name=parser_name)

print(f"파싱 완료: {len(parsed_documents)}개 문서")
for parsed in parsed_documents[:3]:  # 처음 3개만 미리보기
    print(f"  - {parsed.source_path.split('/')[-1]}: {len(parsed.raw_text)} 글자")

## 4단계: 청킹

In [ ]:
from src.db_to_llm.ingest.chunk_service import chunk_documents

chunk_size = ingest_config.get("chunk_size", 800)
chunk_overlap = ingest_config.get("chunk_overlap", 100)

chunks = chunk_documents(
    parsed_documents=parsed_documents,
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

print(f"청킹 완료: {len(chunks)}개 청크")
if chunks:
    print(f"\n첫 번째 청크 미리보기:")
    print(f"  chunk_id: {chunks[0].chunk_id}")
    print(f"  텍스트 미리보기: {chunks[0].chunk_text[:100]}...")

## 5단계: 임베딩 생성

In [ ]:
from src.db_to_llm.ingest.embedding_service import create_embeddings

embedding_model = retrieval_config.get(
    "embedding_model",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print(f"임베딩 모델 로드 중: {embedding_model}")
embedding_items = create_embeddings(chunks=chunks, embedding_model_name=embedding_model)

print(f"임베딩 완료: {len(embedding_items)}개")
if embedding_items:
    print(f"  벡터 차원: {len(embedding_items[0].embedding)}")

## 6단계: ChromaDB 저장 및 확인

In [ ]:
from src.db_to_llm.ingest.vector_store_service import sample_from_collection, upsert_embeddings_to_chroma

chroma_path = str(project_root / retrieval_config.get("chroma_path", "data/chroma"))
collection_name = retrieval_config.get("collection_name", "doc_chunks")

print(f"ChromaDB 저장 시작: path={chroma_path}, collection={collection_name}")
upsert_embeddings_to_chroma(
    embedding_items=embedding_items,
    chroma_path=chroma_path,
    collection_name=collection_name,
)
print("저장 완료!")

# 저장된 데이터 샘플 확인
samples = sample_from_collection(chroma_path=chroma_path, collection_name=collection_name, limit=3)
print(f"\n저장된 데이터 샘플 ({len(samples)}개):")
for sample in samples:
    print(f"  - id: {sample['id'][:20]}..., text: {sample['document'][:60]}...")

## 완료

벡터스토어 구축이 완료되었습니다.
이제 `notebooks/stream/02_end_to_end_graph_flow.ipynb`에서 RAG 검색을 포함한 전체 그래프 흐름을 테스트할 수 있습니다.